# Run your own script

We assume you already have a script for a DP algorithm you want to benchmark ASAP.
We walk through how to wrap it in this library's `BaseAlgorithm` class, install it in a collection, and run a short sweep over it to get recorded results.

## Setup

Here, we wrap a simple script for a DP linear regression algorithm.
The script file we use is [`my_regression_script.py`](my_regression_script.py) (which you can download to run this notebook yourself),
and contains the single function `noisy_regression(X, y, epsilon, x_bound, y_bound, ridge, seed=...)`
which outputs an estimate of the regression coefficients. It requires two pieces of public information: `x_bound` and `y_bound`, which are bounds on the norms of the rows of `X` and `y`, respectively. It also has one hyperparameter, `ridge`, a
regularization term added to the noisy statistics before solving.


## Wrap it in the BaseAlgorithm class

In order for an algorithm to be benchmarked in this library, it must be wrapped in a class which inherits from [`BaseAlgorithm`](../../reference/algorithm/dpstats.algorithm.BaseAlgorithm/).
Implementing this class requires two things: implementing a `run` function and declaring any appropriate hyperparameters.

Recall that `noisy_regression` has the one `ridge` hyperparameter, which we
implement as a class attribute:
```python
ridge: float = parameter_field(
    default=1.0,
    ge=0.0,
    description="Ridge term added to the noisy X'X before solving.",
)
```

The `run` method is where the algorithm is actually executed. It has the following signature:
```python
def run(self, metadata, input, budget, seed)
```
In order to run the script, we simply need to unpack `metadata`, `input`, and `budget` to pass the appropriate arguments to `noisy_regression`.
Recall that `noisy_regression` requires the following inputs:
1. the sensitive data `X` and `y`, which are stored as `input.X` and `input.y`, respectively (`input` is a `LinearRegressionInput`; see the [task definition](../../../linear_regression/definition/)).
2. the l2 norm bounds on the domain of `X` and `y`. `metadata` (a `LinearRegressionMetadata`, same page) contains a [`Domain`](../../reference/domains/dpstats.domains.Domain/) object for each of these, whose `l2_norm_bound` gives the bound. Our script needs both bounds to be finite, so if either is not we raise [`InvalidRegime`](../../reference/exceptions/dpstats.exceptions.InvalidRegime/), which tells the benchmark that this problem is outside what the algorithm can do.
3. the `epsilon` parameter, which we obtain from the `budget` object. `noisy_regression` requires a pure-DP budget, which `budget` (a [`DPBudget`](../../reference/budgets/dpstats.budgets.DPBudget/)) need not be, so we convert it using [`accounting.spend_as_pure`](../../reference/budgets/dpstats.budgets.accounting.spend_as_pure/), which returns a pure-DP budget whose guarantee is sufficient for the offered budget. For example, when offered (ε, δ)-DP, it uses ε-pure-DP and leaves δ unused.

Putting all of this together results in [`my_regression.py`](my_regression.py), which is shown below and which you can download alongside the script.
(For the sake of brevity, the type annotations are left out of this version; the type-annotated copy in [`my_regression_annotated.py`](my_regression_annotated.py) will make coding and implementation easier in an editor.)

The wrapper does not verify a privacy proof. Your script still needs a valid privacy argument for the declared assumptions and budget; budget conversion alone does not establish one. The library uses the standard privacy interpretation unless the algorithm declares otherwise. If your script has regimes where it is not strictly private, or is a deliberately nonprivate control, declare that with the `privacy_interpretation` property (see [privacy guarantees and labels](https://saminulh.github.io/dpstats/library/reference/algorithm/dpstats.algorithm.PrivacyInterpretation/)).


In [ ]:
from IPython.display import Code

Code(filename="my_regression.py", language="python")

## Construct it and check its identity

We can now import the wrapper class and construct an instance of it with a specified hyperparameter.
Each configured algorithm has a UUID, accessible via the `.uuid` attribute, which identifies it in the benchmark results.


In [ ]:
from my_regression import MyRegression

default_algorithm = MyRegression(label="DefaultWrappedScript")
print(f"default_algorithm UUID: {default_algorithm.uuid}, ridge: {default_algorithm.ridge}")


Algorithms are considered equal, and have the same UUID, if they have the same class and the same parameter values; the label is not part of the identity.


In [ ]:
another_algorithm = MyRegression(label="AnotherWrappedScript", ridge=10.0)
default_algorithm_again = MyRegression(label="DefaultAgain", ridge=1.0)

print(f"another_algorithm UUID: {another_algorithm.uuid}, ridge: {another_algorithm.ridge}")
print(f"default_algorithm_again UUID: {default_algorithm_again.uuid}, ridge: {default_algorithm_again.ridge}")
print(f"{default_algorithm.label} and {default_algorithm_again.label} are the same: {default_algorithm == default_algorithm_again}")


## Run it on a few instances

Open the workspace from [Install and fetch the benchmark](install_and_fetch.ipynb) and its linear-regression collection. Fetching again reuses the installed release. We select two small instances with finite public bounds and one with an unbounded domain, so the results show both a successful run and an expected refusal.

The configurations are recorded in your local registry. The next notebook compares them with released methods under the release's sweep.

In [ ]:
from pathlib import Path

from dpstats import fetch_benchmark, open_collection
from dpstats.tasks import LinearRegression

workspace = fetch_benchmark(Path("dp-benchmark"))

In [ ]:
TASK = LinearRegression
collection = open_collection(workspace, TASK)

import numpy as np

def is_bounded(instance):
    metadata = instance.metadata
    return np.isfinite(metadata.design_domain.l2_norm_bound) and np.isfinite(
        metadata.response_domain.l2_norm_bound
    )


by_size = sorted(collection.load_instances().values(), key=lambda inst: inst.metadata.n_samples)
handful = [inst for inst in by_size if is_bounded(inst)][:2]
handful += [inst for inst in by_size if not is_bounded(inst)][:1]
for instance in handful:
    bounds = "bounded" if is_bounded(instance) else "unbounded"
    print(f"n={instance.metadata.n_samples:6d} d={instance.metadata.dim:3d} {bounds:9s} {instance}")

for algorithm in (default_algorithm, another_algorithm):
    print("installed", collection.add_algorithm(algorithm, registry="mine"))

In [ ]:
from dpstats.budgets import PureDPBudget

results = collection.run_sweep(
    algorithms=[default_algorithm, another_algorithm],
    instances=handful,
    budgets=[PureDPBudget(eps=3.0)],
    seeds=[0, 1, 2],
    time_limit=None,
    memory_limit_gb=None,
)
print(f"{len(results)} attempts, statuses: {results['status'].value_counts().to_dict()}")
import pandas as pd

by_instance = pd.crosstab(
    results["instance"].map(str),
    results["status"].map(lambda status: getattr(status, "value", status)),
)
print(by_instance)
display(results.head(1))


## Read the refusal

Our wrapper needs finite design and response bounds. An unbounded instance therefore produces `invalid_regime`, regardless of the `ridge` setting. Inspect the declared bounds before changing the algorithm's parameters.

[Transformations and composition](extend_with_transformations.ipynb) explains how a private preparation stage can supply missing bounds. [Domains](domains.ipynb) explains what the declared bounds mean.

## Saving and resuming a sweep

Use [save_sweep](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.save_sweep) and [load_sweep](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.load_sweep) to keep a sweep declaration for later. [get_sweep_status](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.get_sweep_status) previews completed and pending cells; [run_sweep](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.run_sweep) documents resume and retry policies.

If you provide a concurrent `schedule_cells` adapter, give it an explicit concurrency limit and enforce resource limits in that adapter. Keep one writer per collection.


## Compare with other methods

Continue with [Compare on the benchmark](compare_on_the_benchmark.ipynb) to read status coverage, baseline-relative errors, and profile curves alongside released results.